# Exploratory Data Analysis: Dimuon Angular Distributions

This notebook extracts, analyzes, and visualizes the projected angular observables ($\theta_{XZ}$, $\theta_{YZ}$, and $\Delta\theta$) comparing **Signal** (direct charm decay: $\nu_\mu \to c \to \mu$) vs **Signal-Like** ($2\mu$ traversing DS) across 4 sequential stages of event selection and reconstruction:

1. **Stage 1: All Dimuon Events** (MC truth angles)
2. **Stage 2: Dimuons in DS Acceptance** (MC truth angles)
3. **Stage 3: Dimuons in DS Acceptance with $\ge 2$ Reconstructed DS Hough Tracks** (MC truth angles)
4. **Stage 4: Dimuons in DS Acceptance with $\ge 2$ Reconstructed DS Hough Tracks** (Reconstructed track angles: $\theta^{\mathrm{reco}}$ and $\Delta\theta^{\mathrm{reco}}$)

All conversions use `dur.converters.to_numpy` from `ddfUtils.root` directly. Plots compare **Signal** (solid black step) vs **Signal-Like** (tab:red markers) with clean legends and no floating text.

In [ ]:
import os
import sys
import math
from typing import Dict, List, Tuple, Optional, Union

import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetOptTitle(0)

# Setup ddfUtils
_ddf_path = '/afs/cern.ch/user/i/idioniso/ddfUtils'
if _ddf_path not in sys.path and os.path.exists(_ddf_path):
    sys.path.insert(0, _ddf_path)

import ddfUtils.root as dur
import ddfUtils.mpl_styling as dms
hep.style.use(hep.style.ROOT)
print('Loaded ddfUtils.root (dur) and ddfUtils.mpl_styling (dms) with ROOT style.')


## 1. Histogram Lookup Utility

Helper function to recursively locate a histogram by name within a ROOT directory hierarchy.
Conversions to NumPy arrays use `dur.converters.to_numpy` directly from `ddfUtils.root`.

In [ ]:
def find_histogram(tfile: ROOT.TFile, hname: str) -> Optional[ROOT.TH1]:
    """Recursively finds a TH1 histogram by name within a ROOT TFile."""
    obj = tfile.Get(hname)
    if obj and isinstance(obj, ROOT.TH1):
        return obj

    def scan_dir(d):
        for k in d.GetListOfKeys():
            o = d.Get(k.GetName())
            if isinstance(o, ROOT.TDirectory):
                res = scan_dir(o)
                if res:
                    return res
            elif isinstance(o, ROOT.TH1) and k.GetName() == hname:
                return o
        return None

    return scan_dir(tfile)


## 2. Stage Plotting Function

`plot_stage_angular_distributions` plots the full 6-panel angular grid for a selected stage:
- **Pad 1 & 2**: $\theta_{XZ}(\mu_1)$ and $\theta_{XZ}(\mu_2)$ (log scale)
- **Pad 3 & 4**: $\theta_{YZ}(\mu_1)$ and $\theta_{YZ}(\mu_2)$ (log scale)
- **Pad 5 & 6**: $\Delta\theta_{XZ}$ and $\Delta\theta_{YZ}$ (linear scale)

In [ ]:
STAGE_CONFIGS = {
    1: {
        'title': 'Stage 1: All Dimuon Events (MC Truth)',
        'sig_pref': 'direct',
        'bkg_pref': 'signal_like',
        'is_reco': False,
    },
    2: {
        'title': 'Stage 2: Dimuons in DS Acceptance (MC Truth)',
        'sig_pref': 'direct_in_acc',
        'bkg_pref': 'signal_like_in_acc',
        'is_reco': False,
    },
    3: {
        'title': 'Stage 3: In DS Acc + >=2 DS Tracks (MC Truth)',
        'sig_pref': 'direct_2ds_in_acc',
        'bkg_pref': 'signal_like_2ds_in_acc',
        'is_reco': False,
    },
    4: {
        'title': 'Stage 4: In DS Acc + >=2 DS Tracks (Reconstructed)',
        'sig_pref': 'direct_2ds_in_acc',
        'bkg_pref': 'signal_like_2ds_in_acc',
        'is_reco': True,
    },
}


def plot_stage_angular_distributions(
    input_file: Union[str, ROOT.TFile],
    stage: int = 1,
    sig_key: Optional[str] = None,
    bkg_key: Optional[str] = None,
    sig_label: str = 'numu -> c -> mu',
    bkg_label: str = 'Signal-Like (2 muons traversing DS)',
    figsize: Tuple[int, int] = (15, 12),
):
    """
    Reads ROOT file, extracts angular distributions for the specified stage using
    dur.converters.to_numpy, and plots all 6 distributions together in a 3x2 grid.
    """
    if stage not in STAGE_CONFIGS:
        raise ValueError(f'Invalid stage {stage}. Available: 1, 2, 3, 4')

    cfg = STAGE_CONFIGS[stage]
    spref = sig_key if sig_key else cfg['sig_pref']
    bpref = bkg_key if bkg_key else cfg['bkg_pref']
    is_reco = cfg['is_reco']
    stage_title = cfg['title']

    should_close = False
    if isinstance(input_file, str):
        f = ROOT.TFile.Open(input_file, 'READ')
        should_close = True
    else:
        f = input_file

    if not f or f.IsZombie():
        raise FileNotFoundError(f'Could not open input ROOT file: {input_file}')

    var_pfx = 'reco_' if is_reco else ''
    theta_sym = r'\theta^{\mathrm{reco}}' if is_reco else r'\theta'
    delta_sym = r'\Delta\theta^{\mathrm{reco}}' if is_reco else r'\Delta\theta'

    observables = [
        # (var_name, x_label, is_log, x_lim, y_lim)
        (f'{var_pfx}mu1_theta_xz_mrad', f'${theta_sym}_{{XZ}}(\mu_{{1}})$ [mrad]', True, (-300, 300), (0.5, 2e3)),
        (f'{var_pfx}sec_mu_theta_xz_mrad', f'${theta_sym}_{{XZ}}(\mu_{{2}})$ [mrad]', True, (-300, 300), (0.5, 5e2)),
        (f'{var_pfx}mu1_theta_yz_mrad', f'${theta_sym}_{{YZ}}(\mu_{{1}})$ [mrad]', True, (-300, 300), (0.5, 2e3)),
        (f'{var_pfx}sec_mu_theta_yz_mrad', f'${theta_sym}_{{YZ}}(\mu_{{2}})$ [mrad]', True, (-300, 300), (0.5, 5e2)),
        (f'{var_pfx}dimuon_opening_angle_xz_mrad', f'${delta_sym}_{{XZ}}$ [mrad]', False, (0, 300), (0, None)),
        (f'{var_pfx}dimuon_opening_angle_yz_mrad', f'${delta_sym}_{{YZ}}$ [mrad]', False, (0, 300), (0, None)),
    ]

    fig, axes = plt.subplots(3, 2, figsize=figsize)
    fig.suptitle(stage_title, fontsize=16, fontweight='bold', y=0.99)

    for idx, (var, xlabel, is_log, xlim, ylim) in enumerate(observables):
        ax = axes[idx // 2, idx % 2]
        h_sig = find_histogram(f, f'h_{spref}_{var}')
        h_bkg = find_histogram(f, f'h_{bpref}_{var}')

        if h_sig is None or h_bkg is None:
            ax.text(0.5, 0.5, f'Missing: {var}', ha='center', va='center', transform=ax.transAxes)
            continue

        # Use ddfUtils dur.converters.to_numpy
        centers_sig, counts_sig, edges_sig, _, errs_sig = dur.converters.to_numpy(h_sig)
        centers_bkg, counts_bkg, edges_bkg, _, errs_bkg = dur.converters.to_numpy(h_bkg)
        n_sig = int(h_sig.GetEntries())
        n_bkg = int(h_bkg.GetEntries())

        # Plot Signal: Solid Black Step line
        ax.step(
            edges_sig,
            np.append(counts_sig, counts_sig[-1]),
            where='post',
            color='black',
            linewidth=1.8,
            label=f'{sig_label} (N={n_sig})',
        )

        # Plot Background / Signal-Like: Red markers with error bars
        mask = counts_bkg > 0 if is_log else slice(None)
        ax.errorbar(
            centers_bkg[mask],
            counts_bkg[mask],
            yerr=errs_bkg[mask],
            fmt='o',
            color='tab:red',
            ecolor='tab:red',
            markersize=4,
            capsize=2,
            elinewidth=1.2,
            label=f'{bkg_label} (N={n_bkg})',
        )

        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel('Events', fontsize=12)
        ax.set_xlim(xlim)

        if is_log:
            ax.set_yscale('log')
            max_val = max(counts_sig.max() if len(counts_sig) else 1,
                          counts_bkg.max() if len(counts_bkg) else 1)
            ax.set_ylim(0.5, max(max_val * 4.0, 10.0))
        else:
            max_val = max(counts_sig.max() if len(counts_sig) else 1,
                          counts_bkg.max() if len(counts_bkg) else 1)
            ax.set_ylim(0, max_val * 1.3)

        ax.legend(frameon=False, loc='upper right', fontsize=10)
        ax.grid(True, which='both', linestyle='--', alpha=0.5)

    plt.tight_layout()
    if should_close:
        f.Close()
    return fig, axes


## 3. Comparing Across All 4 Stages

`plot_all_stages_evolution` plots how a specific observable (e.g. $\Delta\theta_{XZ}$ or $\theta_{XZ}(\mu_1)$) evolves from Stage 1 to Stage 4 side-by-side.

In [ ]:
def plot_all_stages_evolution(
    input_file: Union[str, ROOT.TFile],
    observable: str = 'dimuon_opening_angle_xz_mrad',
    obs_label: str = r'$\Delta\theta_{XZ}$ [mrad]',
    is_log: bool = False,
    xlim: Tuple[float, float] = (0, 300),
    sig_key: str = 'direct',
    bkg_key: str = 'signal_like',
    sig_label: str = 'numu -> c -> mu',
    bkg_label: str = 'Signal-Like',
    figsize: Tuple[int, int] = (18, 4),
):
    """Compares the distribution of an observable across the 4 stages side-by-side."""
    should_close = False
    if isinstance(input_file, str):
        f = ROOT.TFile.Open(input_file, 'READ')
        should_close = True
    else:
        f = input_file

    if not f or f.IsZombie():
        raise FileNotFoundError(f'Could not open input ROOT file: {input_file}')

    fig, axes = plt.subplots(1, 4, figsize=figsize, sharey=False)

    for stage_idx in range(1, 5):
        ax = axes[stage_idx - 1]
        cfg = STAGE_CONFIGS[stage_idx]
        spref = f"{sig_key}_{cfg['sig_pref'].split('_', 1)[1]}" if '_' in cfg['sig_pref'] else sig_key
        bpref = f"{bkg_key}_{cfg['bkg_pref'].split('_', 1)[1]}" if '_' in cfg['bkg_pref'] else bkg_key
        var_pfx = 'reco_' if cfg['is_reco'] else ''

        h_sig = find_histogram(f, f'h_{spref}_{var_pfx}{observable}')
        h_bkg = find_histogram(f, f'h_{bpref}_{var_pfx}{observable}')

        ax.set_title(cfg['title'], fontsize=11, fontweight='bold')
        if h_sig and h_bkg:
            centers_sig, counts_sig, edges_sig, _, _ = dur.converters.to_numpy(h_sig)
            centers_bkg, counts_bkg, edges_bkg, _, errs_bkg = dur.converters.to_numpy(h_bkg)
            n_sig = int(h_sig.GetEntries())
            n_bkg = int(h_bkg.GetEntries())

            ax.step(
                edges_sig,
                np.append(counts_sig, counts_sig[-1]),
                where='post',
                color='black',
                linewidth=1.8,
                label=f'{sig_label} ({n_sig})',
            )

            mask = counts_bkg > 0 if is_log else slice(None)
            ax.errorbar(
                centers_bkg[mask],
                counts_bkg[mask],
                yerr=errs_bkg[mask],
                fmt='o',
                color='tab:red',
                markersize=4,
                capsize=2,
                label=f'{bkg_label} ({n_bkg})',
            )
            ax.legend(frameon=False, fontsize=9)

        ax.set_xlabel(obs_label, fontsize=11)
        ax.set_xlim(xlim)
        if is_log:
            ax.set_yscale('log')
            ax.set_ylim(bottom=0.5)
        ax.grid(True, linestyle='--', alpha=0.5)

    axes[0].set_ylabel('Events', fontsize=12)
    plt.tight_layout()
    if should_close:
        f.Close()
    return fig, axes


## 4. Run Angular Comparisons

Open `/eos/user/i/idioniso/snd-numu-charm/output_nu14_all_hists.root` and generate plots for Stages 1 to 4.

In [ ]:
INPUT_FILE = "/eos/user/i/idioniso/snd-numu-charm/output_nu14_all_hists.root"
tfile = ROOT.TFile.Open(INPUT_FILE, "READ") if os.path.exists(INPUT_FILE) else None
if tfile and not tfile.IsZombie():
    print(f"Successfully opened {INPUT_FILE}")
else:
    print(f"Failed to open {INPUT_FILE}")


In [ ]:
# Plot Stage 1: All Dimuon Events (theta_XZ, theta_YZ, and Delta theta)
if tfile:
    fig1, axes1 = plot_stage_angular_distributions(
        input_file=tfile,
        stage=1,
        sig_label="numu -> c -> mu",
        bkg_label="Signal-Like (2 muons traversing DS)",
    )
    plt.show()


In [ ]:
# Plot Stage 2: In DS Acceptance
if tfile:
    fig2, axes2 = plot_stage_angular_distributions(
        input_file=tfile,
        stage=2,
        sig_label="numu -> c -> mu",
        bkg_label="Signal-Like (2 muons traversing DS)",
    )
    plt.show()


In [ ]:
# Plot Stage 3: In DS Acceptance + >=2 DS Tracks (MC Truth Angles)
if tfile:
    fig3, axes3 = plot_stage_angular_distributions(
        input_file=tfile,
        stage=3,
        sig_label="numu -> c -> mu",
        bkg_label="Signal-Like (2 muons traversing DS)",
    )
    plt.show()


In [ ]:
# Plot Stage 4: In DS Acceptance + >=2 DS Tracks (Reconstructed Track Angles)
if tfile:
    fig4, axes4 = plot_stage_angular_distributions(
        input_file=tfile,
        stage=4,
        sig_label="numu -> c -> mu",
        bkg_label="Signal-Like (2 muons traversing DS)",
    )
    plt.show()


In [ ]:
# Compare Stage 1 vs Stage 2 vs Stage 3 vs Stage 4 for Opening Angle Delta theta_XZ & Delta theta_YZ
if tfile:
    fig_evo_xz, axes_evo_xz = plot_all_stages_evolution(
        input_file=tfile,
        observable="dimuon_opening_angle_xz_mrad",
        obs_label=r"$\Delta\theta_{XZ}$ [mrad]",
        is_log=False,
        xlim=(0, 300),
    )
    plt.show()

    fig_evo_yz, axes_evo_yz = plot_all_stages_evolution(
        input_file=tfile,
        observable="dimuon_opening_angle_yz_mrad",
        obs_label=r"$\Delta\theta_{YZ}$ [mrad]",
        is_log=False,
        xlim=(0, 300),
    )
    plt.show()
